# BitSemCom on Colab: training the original model with a fixed schedule

Runs section 6 of `README.md` end to end: `train_bitsemcom.py` trains the original BitSemCom model (same network,
bit mapper, QPSK + AWGN channel and MSE loss as `main.py`) for a fixed number of steps with a warmup + cosine
learning-rate schedule. Everything persistent lives in Google Drive (`MyDrive/BitSemCom`): the pretrained model you
upload once, the data zips and the training runs. The code is cloned fresh from GitHub onto the VM's local disk
every session.

**Before the first run**

1. Drive: create `MyDrive/BitSemCom/Pretrain/` and upload the model you start from, for example
   `BitSemCom_C32_stage2_ep480.model` (Drive website).
2. Runtime: *Runtime > Change runtime type > GPU*. A100 or L4 (bf16); T4 works but is slower (fp16).
3. Only if the GitHub repository is private: *GitHub > Settings > Developer settings > Fine-grained personal access
   tokens*, new token with repository access to `BitSemCom` and permission *Contents: read*. In Colab, key icon in
   the left sidebar, add a secret named `GH_TOKEN` with that token and switch on *Notebook access*.

**Every session**: run cells 1 to 4 (parameters, Drive and code, packages, data), then the training cell. Training is
resumable: if the session dies or the time budget is reached, run the same cell again.

## 1. Parameters

In [ ]:
GITHUB_REPO = "SafaaSalman/BitSemCom"
BRANCH = "bitsemcom-original"

DRIVE = "/content/drive/MyDrive/BitSemCom"   # persistent: Pretrain/, data/, runs/
REPO = "/content/BitSemCom"                  # fresh clone on the VM's local disk (fast)
RUNS = DRIVE + "/runs"

C = 32
INIT = "Pretrain/BitSemCom_C32_stage2_ep480.model"   # model to start from ("none" = random weights)
RUN_NAME = f"C{C}_bench"
STEPS, BATCH = 30000, 16
MAX_HOURS = 11                               # save and stop before the Colab session limit; rerun the cell to continue
SNR_MIN, SNR_MAX = 0, 10                     # training Es/N0 range (dB), drawn per batch

## 2. Google Drive and the code

If the clone fails, clear this cell's output before sharing the notebook: git may echo the URL that carries the token.

In [ ]:
import os, glob
from google.colab import drive, userdata

drive.mount("/content/drive")
for d in ("Pretrain", "data", "runs"):
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)

try:
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""                               # no secret: works when the repository is public
URL = f"https://{TOKEN}@github.com/{GITHUB_REPO}" if TOKEN else f"https://github.com/{GITHUB_REPO}"

if not os.path.isdir(f"{REPO}/.git"):
    !git clone -q -b {BRANCH} {URL} {REPO}
    !git -C {REPO} remote set-url origin https://github.com/{GITHUB_REPO}
else:
    !git -C {REPO} pull -q {URL} {BRANCH}
assert os.path.isdir(f"{REPO}/.git"), "clone failed: for a private repository add the GH_TOKEN secret (see the top of the notebook)"
%cd {REPO}
!git log --oneline -1

## 3. Packages and GPU

In [ ]:
!pip -q install timm
import os, torch
from IPython.display import Image as IMG, display

assert torch.cuda.is_available(), "select a GPU runtime: Runtime > Change runtime type"
AMP = "bf16" if torch.cuda.get_device_capability(0)[0] >= 8 else "fp16"   # bf16 on A100 / L4 / H100, fp16 on T4
print(torch.cuda.get_device_name(0), "| autocast", AMP, "| torch", torch.__version__)

## 4. Pretrained model and data

Downloads happen once and are kept on Drive (`data/`). DIV2K is unzipped to the local disk every session because
reading 800 large PNGs through the Drive mount is slow. The 24 Kodak 256x256 centre crops are the validation set.

In [ ]:
import os, glob
from PIL import Image

if INIT != "none":
    if not os.path.exists(f"{DRIVE}/{INIT}"):
        raise SystemExit(f"upload {os.path.basename(INIT)} to {DRIVE}/Pretrain/ first (Drive website)")
    os.makedirs("Pretrain", exist_ok=True)
    !cp {DRIVE}/Pretrain/*.model Pretrain/
    !ls -la Pretrain

DIV2K_ZIP = f"{DRIVE}/data/DIV2K_train_HR.zip"       # 800 images, 3.5 GB
if os.path.exists(DIV2K_ZIP) and os.path.getsize(DIV2K_ZIP) < 3_000_000_000:
    print("DIV2K zip on Drive is incomplete, downloading again"); os.remove(DIV2K_ZIP)
if not os.path.exists(DIV2K_ZIP):
    !wget -q --show-progress -O /content/DIV2K_train_HR.zip https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
    !unzip -tq /content/DIV2K_train_HR.zip && cp /content/DIV2K_train_HR.zip {DIV2K_ZIP}
if not os.path.isdir("datasets/DIV2K_train_HR"):
    !unzip -q {DIV2K_ZIP} -d datasets/
print("DIV2K images:", len(glob.glob("datasets/DIV2K_train_HR/*.png")))

KODAK = f"{DRIVE}/data/kodak_full"
os.makedirs(KODAK, exist_ok=True)
for i in range(1, 25):
    f = f"{KODAK}/kodim{i:02d}.png"
    url = f"https://r0k.us/graphics/kodak/kodak/kodim{i:02d}.png"
    if not os.path.exists(f):
        !wget -q -O {f} {url}
!python experiments/make_kodak_patches.py --src {KODAK} --dst datasets/kodak_patches256

## 5. Training

AdamW, 500 warmup steps then cosine 1e-4 to 1e-6 over the step budget, per-batch SNR drawn uniformly in
[SNR_MIN, SNR_MAX], validation on the Kodak patches every 1000 steps, best and last checkpoints. No early stopping:
with the 1e-4 peak the validation score dips below its starting point for the first thousands of steps and recovers
only as the learning rate decays, so a fixed step budget is used. 30k steps took 5.8 h on an L4.
Resumable: rerun the cell.

In [ ]:
OUT = f"{RUNS}/{RUN_NAME}"
!python train_bitsemcom.py --C {C} --init {INIT} --train_dir datasets/DIV2K_train_HR --val_dir datasets/kodak_patches256 --steps {STEPS} --batch {BATCH} --lr 1e-4 --min_lr 1e-6 --warmup 500 --snr_min {SNR_MIN} --snr_max {SNR_MAX} --val_every 1000 --patience 0 --amp_dtype {AMP} --num_workers 4 --max_hours {MAX_HOURS} --resume --out {OUT}

## 6. Training status and curves

`noiseless` = the decoder receives the exact bits; `uncoded_soft` = the bits went through the QPSK + AWGN channel and
the soft demodulator (the normal BitSemCom receiver).

In [ ]:
import csv, glob, os
runs = [r for r in sorted(glob.glob(f"{RUNS}/C{C}_*")) if os.path.exists(f"{r}/log.csv")]
for run in runs:
    rows = list(csv.DictReader(open(f"{run}/val.csv"))) if os.path.exists(f"{run}/val.csv") else []
    if rows:
        last = rows[-1]["step"]
        print(os.path.basename(run), "| last validation at step", last, "|",
              " ".join("%s@%s=%.2f" % (r["receiver"][:8], r["snr"], float(r["psnr"])) for r in rows if r["step"] == last))
RUN_ARGS = " ".join(runs)
!python experiments/plot_training.py {RUN_ARGS} --out {RUNS}
display(IMG(f"{RUNS}/training.png", width=1100))

## 7. Where the model is

The trained model is `MyDrive/BitSemCom/runs/<RUN_NAME>/best.model` (weights with the best validation score);
`last.model` is the state at the last validation. To test it the original way, see section 7 of `README.md`
(`main.py --pretrain <path to best.model> --test_snr 6`).